In [ ]:
pip install plotly

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd

In [ ]:
warehouses_path = r'C:\Users\titas\Desktop\LP_Project\Warehouse Optimisation\data\warehouses.csv'
stores_path = r'C:\Users\titas\Desktop\LP_Project\Warehouse Optimisation\data\stores.csv'


In [ ]:
df = pd.read_csv(warehouses_path)
df
df_stores = pd.read_csv(stores_path)
df_stores

In [ ]:
# convert long/lat to points
df_geo = gpd.GeoDataFrame(df, geometry = gpd.points_from_xy(df.longitude, df.latitude))
df_geo_stores = gpd.GeoDataFrame(df_stores, geometry = gpd.points_from_xy(df_stores.longitude, df_stores.latitude))
df_geo

In [ ]:
%pip install geodatasets
import geodatasets

# get built-in dataset
world_data = gpd.read_file(geodatasets.get_path('naturalearth.land'))
world_data

In [ ]:
# plot the UK map with warehouses and stores
axis = world_data.plot(color='lightblue', edgecolor='black')
df_geo.plot(ax=axis, color='red', markersize=70, label='Warehouses')
df_geo_stores.plot(ax=axis, color='blue', markersize=35, label='Stores')

# Zoom the map view exclusively to the UK coordinates
axis.set_xlim(-7, 2)   # Longitude (West to East)
axis.set_ylim(49, 60)   # Latitude (South to North)

plt.title('UK Warehouses and Stores')
axis.legend()

fig = plt.gcf()
fig.set_size_inches(9,6)
fig.savefig('matplot.png', dpi = 200)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

def haversine_cross(df1, df2, lat_col='latitude', lon_col='longitude', R=6371):
    lat1 = np.radians(df1[lat_col].values)[:, None]   # shape (n1, 1)
    lon1 = np.radians(df1[lon_col].values)[:, None]
    lat2 = np.radians(df2[lat_col].values)[None, :]   # shape (1, n2)
    lon2 = np.radians(df2[lon_col].values)[None, :]

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

dist_matrix = haversine_cross(df_geo, df_stores)

dist_df = pd.DataFrame(
    dist_matrix,
    index=df_geo['name'],
    columns = df_stores['name']
)


In [ ]:
dist_df

In [ ]:
# plot the UK map with warehouses and stores
axis = world_data.plot(color='lightblue', edgecolor='black')
df_geo.plot(ax=axis, color='red', markersize=70, label='Warehouses')
df_geo_stores.plot(ax=axis, color='blue', markersize=35, label='Stores')

# draw a line for every warehouse-store pair
for _, wh in df_geo.iterrows():
    for _, store in df_geo_stores.iterrows():
        axis.plot(
            [wh['longitude'], store['longitude']],
            [wh['latitude'], store['latitude']],
            color='yellow', linewidth=0.5, alpha=0.7, zorder=1
        )

axis.set_xlim(-7, 2)
axis.set_ylim(49, 60)
plt.title('UK Warehouses and Stores')
axis.legend()
fig = plt.gcf()
fig.set_size_inches(9, 6)
fig.savefig('matplot.png', dpi=200)
plt.show()

In [ ]:
dist_df

In [ ]:
for row_label, row_data in dist_df.iterrows():
   
    print(row_data)
    

In [ ]:
basildon_distance = dist_df.loc['Basildon Distribution Centre']
basildon_df = basildon_distance.to_frame()
basildon_df = basildon_df.rename(columns={'Basildon Distribution Centre': 'distance'})
basildon_df['upper_cost'] = basildon_df['distance'] * 0.55
basildon_df['lower_cost'] = basildon_df['distance'] * 0.35

In [ ]:
basildon_df

In [ ]:
birmingham_distance = dist_df.loc['Birmingham Distribution Centre']
birmingham_df = birmingham_distance.to_frame()
birmingham_df = birmingham_df.rename(columns={'Birmingham Distribution Centre': 'distance'})
birmingham_df['upper_cost'] = birmingham_df['distance'] * 0.55
birmingham_df['lower_cost'] = birmingham_df['distance'] * 0.3


In [ ]:
birmingham_df

In [ ]:
leeds_distance = dist_df.loc['Leeds Distribution Centre']
leeds_df = leeds_distance.to_frame()
leeds_df = leeds_df.rename(columns={'Leeds Distribution Centre': 'distance'})
leeds_df['upper_cost'] = leeds_df['distance'] * 0.55
leeds_df['lower_cost'] = leeds_df['distance'] * 0.35

In [ ]:
leeds_df

## Standard Form

**Minimise:**

$$
\sum_{i=1}^{3} \sum_{j=1}^{15} c_{ij} x_{ij}
$$

**Subject to:**

$$
\sum_{j=1}^{15} x_{ij} + s_i = \text{capacity}_i
\quad \forall i \in \{1,2,3\} \text{ (warehouses)}
$$

$$
\sum_{i=1}^{3} x_{ij} - e_j = \text{demand}_j
\quad \forall j \in \{1,\ldots,15\} \text{ (stores)}
$$

$$
x_{ij},\, s_i,\, e_j \geq 0
\quad \forall i,j
$$

In [ ]:
pip install pulp

In [ ]:
import pulp as pl

prob = pl.LpProblem("Warehouse_Transportation", pl.LpMinimize)

In [ ]:
warehouses = df_geo['name'].tolist()
stores = df_geo_stores['name'].tolist()

In [ ]:
warehouses, stores

In [ ]:
x = pl.LpVariable.dicts("ship", (warehouses,stores), lowBound=0)

In [ ]:
prob 

In [ ]:
cost_df = pd.concat(
    [basildon_df, birmingham_df, leeds_df],
    keys=['Basildon Distribution Centre', 'Birmingham Distribution Centre', 'Leeds Distribution Centre']
)

In [ ]:
cost_df

In [ ]:
cost_df.loc[('Birmingham Distribution Centre', 'Leicester'), 'upper_cost']

In [ ]:
prob += pl.lpSum([
    cost_df.loc[(i, j), 'upper_cost'] * x[i][j]
    for i in warehouses
    for j in stores
]), "Total Upper Cost"

In [ ]:
for i in warehouses:

    capacity_i = df_geo[df_geo['name'] == i]['capacity'].values[0]

    prob += (
        pl.lpSum([x[i][j] for j in stores]) <= capacity_i,
        f"Capacity_{i}"
    )

In [ ]:
for j in stores:

    demand_j = df_geo_stores[df_geo_stores['name'] == j]['demand'].values[0]


    prob += (
        pl.lpSum([x[i][j] for i in warehouses]) >= demand_j,
        f"Demand_{j}"
    )

In [ ]:
prob.writeLP("Warehouse_Optimisation.lp")

In [ ]:
prob.solve()

In [ ]:
print("Status:", pl.LpStatus[prob.status])

In [ ]:
for v in prob.variables():
    print(v.name, "=", v.varValue)

In [ ]:
print("The total cost of shipping is", pl.value(prob.objective))

In [ ]:
cost_df.xs('London', level='name')['upper_cost']

In [ ]:
for j in stores:
    total_shipped = sum(x[i][j].varValue for i in warehouses)
    demand_j = df_geo_stores[df_geo_stores['name'] == j]['demand'].values[0]
    print(j, total_shipped, demand_j, total_shipped == demand_j)

In [ ]:
for i in warehouses:
    total_shipped = sum(x[i][j].varValue for j in stores)
    capacity_i = df_geo[df_geo['name'] == i]['capacity'].values[0]
    print(i, total_shipped, capacity_i, total_shipped <= capacity_i)

Now time for the lower_bound

In [ ]:
prob_lower = pl.LpProblem("Warehouse_Transportation_Lower", pl.LpMinimize)

In [ ]:
prob_lower += pl.lpSum([
    cost_df.loc[(i, j), 'lower_cost'] * x[i][j]
    for i in warehouses
    for j in stores
]), "Total Lower Cost"

In [ ]:
for i in warehouses:

    capacity_i = df_geo[df_geo['name'] == i]['capacity'].values[0]

    prob_lower += (
        pl.lpSum([x[i][j] for j in stores]) <= capacity_i,
        f"Capacity_{i}"
    )

In [ ]:
for j in stores:

    demand_j = df_geo_stores[df_geo_stores['name'] == j]['demand'].values[0]


    prob_lower += (
        pl.lpSum([x[i][j] for i in warehouses]) >= demand_j,
        f"Demand_{j}"
    )

In [ ]:
prob_lower.writeLP("Warehouse_Optimisation_lower.lp")

In [ ]:
prob_lower.solve()

In [ ]:
print("Status:", pl.LpStatus[prob_lower.status])

In [ ]:
for v in prob_lower.variables():
    print(v.name, "=", v.varValue)

In [ ]:
print("The total lower cost of shipping is", pl.value(prob_lower.objective))

In [ ]:
nearest_warehouse = dist_df.idxmin(axis=0)  # store -> nearest warehouse name

naive_total_cost_upper = 0
naive_total_cost_lower = 0
for j in stores:
    wh = nearest_warehouse[j]
    demand_j = df_geo_stores[df_geo_stores['name'] == j]['demand'].values[0]
    cost_per_unit_upper = cost_df.loc[(wh, j), 'upper_cost']
    cost_per_unit_lower = cost_df.loc[(wh, j), 'lower_cost']
    naive_total_cost_upper += cost_per_unit_upper * demand_j
    naive_total_cost_lower += cost_per_unit_lower * demand_j


In [ ]:
formatted = f"£{naive_total_cost_lower:,.2f}"
print("The naive unoptimised lower bound cost",formatted)
formatted = f"£{naive_total_cost_upper:,.2f}"
print("The naive unoptimised upper bound cost",formatted)

In [ ]:
for i in warehouses:
    naive_used = sum(
        df_geo_stores[df_geo_stores['name']==j]['demand'].values[0]
        for j in stores if nearest_warehouse[j] == i
    )
    capacity_i = df_geo[df_geo['name']==i]['capacity'].values[0]
    print(i, "naive demand assigned:", naive_used, "capacity:", capacity_i, "feasible:", naive_used <= capacity_i)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = ['Lower bound\n(best-case)', 'Upper bound\n(worst-case)']
naive_costs = [naive_total_cost_lower, naive_total_cost_upper]
lp_costs = [pl.value(prob_lower.objective), pl.value(prob.objective)]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 6))
bars_naive = ax.bar(x - width/2, naive_costs, width, label='Naive (nearest-only)', color='indianred')
bars_lp = ax.bar(x + width/2, lp_costs, width, label='LP-Optimised', color='steelblue')

ax.set_ylabel('Total Cost (£)')
ax.set_title('Naive vs Optimised Shipping Cost')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# annotate the naive bars as infeasible
for bar in bars_naive:
    height = bar.get_height()
    ax.annotate('⚠ Infeasible\n(Leeds DC over capacity)',
                xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=8, color='darkred')

# label bar values
for bars in [bars_naive, bars_lp]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'£{h:,.0f}', xy=(bar.get_x()+bar.get_width()/2, h),
                    xytext=(0, -15), textcoords='offset points',
                    ha='center', fontsize=8, color='white')

plt.tight_layout()
plt.savefig('naive_vs_optimised.png', dpi=200)
plt.show()




In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pulp as pl

labels = ['Lower bound\n(best-case)', 'Upper bound\n(worst-case)']
naive_costs = [naive_total_cost_lower, naive_total_cost_upper]
lp_costs = [pl.value(prob_lower.objective), pl.value(prob.objective)]

x_pos = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 6))
bars_naive = ax.bar(x_pos - width/2, naive_costs, width, label='Naive (nearest-only)', color='indianred')
bars_lp = ax.bar(x_pos + width/2, lp_costs, width, label='LP-Optimised', color='steelblue')

ax.set_ylabel('Total Cost (£)')
ax.set_title('Naive vs Optimised Shipping Cost')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.legend()

for bar in bars_naive:
    height = bar.get_height()
    ax.annotate('⚠ Infeasible\n(Leeds DC over capacity)',
                xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=8, color='darkred')

for bars in [bars_naive, bars_lp]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'£{h:,.0f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, -15), textcoords='offset points',
                    ha='center', fontsize=8, color='white')

plt.tight_layout()
plt.savefig('naive_vs_optimised.png', dpi=200)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# colour-code the naive bars: red for violating, muted for compliant
naive_colors = ['indianred' if used <= cap else 'firebrick' 
                 for used, cap in zip(naive_used, capacities)]

bars_naive = ax.bar(x_pos - width/2, naive_used, width, 
                     label='Naive demand assigned', color=naive_colors)
bars_lp = ax.bar(x_pos + width/2, lp_used, width, 
                  label='LP demand assigned', color='steelblue')

ax.plot(x_pos, capacities, 'k--', marker='D', label='Capacity limit')

# shade the "over capacity" zone for each warehouse individually
for xi, cap in zip(x_pos, capacities):
    ax.fill_between([xi - 0.5, xi + 0.5], cap, max(naive_used + [cap])*1.1,
                     color='red', alpha=0.08, zorder=0)

# annotate the overshoot explicitly
for xi, used, cap in zip(x_pos, naive_used, capacities):
    if used > cap:
        ax.annotate(f'⚠ Over by {used - cap:,.0f} units',
                    xy=(xi - width/2, used),
                    xytext=(0, 10), textcoords='offset points',
                    ha='center', fontsize=9, color='darkred', fontweight='bold')

ax.set_xticks(x_pos)
ax.set_xticklabels([w.replace(' Distribution Centre', '') for w in warehouses])
ax.set_ylabel('Units')
ax.set_title('Warehouse Utilisation: Naive vs LP-Optimised')
ax.legend()
plt.tight_layout()
plt.savefig('utilisation_comparison.png', dpi=200)
plt.show()

In [ ]:
print("=== Optimal Shipping Routes ===\n")
for i in warehouses:
    for j in stores:
        qty = x_upper[i][j].varValue
        if qty > 0:
            unit_cost = cost_df.loc[(i, j), 'upper_cost']
            route_cost = qty * unit_cost
            print(f"{i} → {j}: {qty:.0f} units @ £{unit_cost:.2f}/unit = £{route_cost:,.2f}")

print(f"\nTotal routes used: {sum(1 for i in warehouses for j in stores if x_upper[i][j].varValue > 0)} of {len(warehouses)*len(stores)} possible")
print(f"Total cost: £{pl.value(prob.objective):,.2f}")

In [ ]:
## Primal Problem (Standard Form)

**Minimise:**
$$\sum_{i=1}^{3} \sum_{j=1}^{15} c_{ij} x_{ij}$$

**Subject to:**
$$\sum_{j=1}^{15} x_{ij} + s_i = \text{capacity}_i \quad \forall i \in \{1,2,3\} \text{ (warehouses)}$$
$$\sum_{i=1}^{3} x_{ij} - e_j = \text{demand}_j \quad \forall j \in \{1,...,15\} \text{ (stores)}$$
$$x_{ij}, \, s_i, \, e_j \geq 0 \quad \forall i,j$$

---

## Dual Problem

Let $u_i$ be the dual variable associated with warehouse $i$'s capacity constraint, and $v_j$ the dual variable associated with store $j$'s demand constraint.

**Maximise:**
$$\sum_{i=1}^{3} \text{capacity}_i \cdot u_i + \sum_{j=1}^{15} \text{demand}_j \cdot v_j$$

**Subject to:**
$$u_i + v_j \leq c_{ij} \quad \forall i,j$$
$$u_i \leq 0 \quad \forall i \; \text{(dual of a} \leq \text{constraint)}$$
$$v_j \geq 0 \quad \forall j \; \text{(dual of a} \geq \text{constraint)}$$

---

## Economic Interpretation

- $u_i$ is the **shadow price of capacity** at warehouse $i$ — the marginal change in total cost if warehouse $i$'s capacity were relaxed by one unit. It is $\leq 0$ because *more* capacity can only reduce or maintain total cost (never increase it), consistent with a minimisation problem.
- $v_j$ is the **shadow price of demand** at store $j$ — the marginal change in total cost if store $j$'s demand increased by one unit. It is $\geq 0$ because serving *more* demand can only increase or maintain total cost.
- By **complementary slackness**: $u_i \neq 0$ only where warehouse $i$'s capacity constraint is binding (fully utilised) in the optimal primal solution — this held only for **Leeds DC** in this project (utilised at exactly 4,000/4,000 in both the upper and lower cost scenarios), so Leeds DC is the only warehouse with a non-zero shadow price on capacity. Basildon and Birmingham DC, having spare capacity in the optimal solution, have $u_i = 0$.

  Cell In[73], line 3
    **Minimise:**
    ^
SyntaxError: invalid synta
    **Minimise:**
    ^
SyntaxError: invalid syntax

## Primal Problem (Standard Form)

**Minimise:**
$$\sum_{i=1}^{3} \sum_{j=1}^{15} c_{ij} x_{ij}$$

**Subject to:**
$$\sum_{j=1}^{15} x_{ij} + s_i = \text{capacity}_i \quad \forall i \in \{1,2,3\} \text{ (warehouses)}$$
$$\sum_{i=1}^{3} x_{ij} - e_j = \text{demand}_j \quad \forall j \in \{1,...,15\} \text{ (stores)}$$
$$x_{ij}, \, s_i, \, e_j \geq 0 \quad \forall i,j$$

---

## Dual Problem

Let $u_i$ be the dual variable associated with warehouse $i$'s capacity constraint, and $v_j$ the dual variable associated with store $j$'s demand constraint.

**Maximise:**
$$\sum_{i=1}^{3} \text{capacity}_i \cdot u_i + \sum_{j=1}^{15} \text{demand}_j \cdot v_j$$

**Subject to:**
$$u_i + v_j \leq c_{ij} \quad \forall i,j$$
$$u_i \leq 0 \quad \forall i \; \text{(dual of a} \leq \text{constraint)}$$
$$v_j \geq 0 \quad \forall j \; \text{(dual of a} \geq \text{constraint)}$$

---

## Economic Interpretation

- $u_i$ is the **shadow price of capacity** at warehouse $i$ — the marginal change in total cost if warehouse $i$'s capacity were relaxed by one unit. It is $\leq 0$ because *more* capacity can only reduce or maintain total cost (never increase it), consistent with a minimisation problem.
- $v_j$ is the **shadow price of demand** at store $j$ — the marginal change in total cost if store $j$'s demand increased by one unit. It is $\geq 0$ because serving *more* demand can only increase or maintain total cost.
- By **complementary slackness**: $u_i \neq 0$ only where warehouse $i$'s capacity constraint is binding (fully utilised) in the optimal primal solution — this held only for **Leeds DC** in this project (utilised at exactly 4,000/4,000 in both the upper and lower cost scenarios), so Leeds DC is the only warehouse with a non-zero shadow price on capacity. Basildon and Birmingham DC, having spare capacity in the optimal solution, have $u_i = 0$.